# 딥러닝응용III — Lecture 03 실습
# 토큰화 기초: BPE 어휘 집합 구축과 GPT 입력값 생성 (답안 포함)

| 항목 | 내용 |
|------|------|
| **강의** | 딥러닝응용III (자연어처리) |
| **수업** | 3주차 — 토큰화 기초 |
| **전공** | 데이터사이언스전공, 동덕여자대학교 |
| **교수** | 유원상 교수 |
| **참고** | 이기창, *BERT와 GPT로 배우는 자연어 처리* (이지스퍼블리싱) |

> 이 노트북은 **교수용**입니다. 모든 문제의 답안과 풀이가 포함되어 있습니다.

---

### 🎯 학습 목표

1. **서브워드 토큰화의 필요성** 이해
2. **BPE 어휘 집합 구축 과정** 이해
3. **GPT 입력값 생성** — `input_ids`와 `attention_mask`의 의미와 생성 방법 이해
4. **Byte-Level BPE의 한국어 표현 원리** — UTF-8 바이트와 `byte_encoder` 매핑 이해

---

## 셀 1. 실습 환경 만들기

### ⚙️ 런타임 설정

상단 메뉴 → **런타임** → **런타임 유형 변경** → 하드웨어 가속기 → **None (CPU)**

> 이 실습은 토크나이저 학습과 텍스트 전처리만 하므로 GPU가 필요 없습니다.

### 📦 설치하는 패키지 3가지

| 패키지 | 역할 |
|--------|------|
| `korpora` | 한국어 공개 말뭉치 로더. `Korpora.load("nsmc")`로 NSMC를 자동 다운로드 |
| `tokenizers` | HuggingFace 저수준 토크나이저. BPE 어휘 집합 **학습 자체**를 수행 (Rust 구현) |
| `transformers` | HuggingFace 고수준 라이브러리. 학습된 토크나이저를 `GPT2Tokenizer`로 감싸 입력값 생성 |

> 💡 **핵심**: `tokenizers`로 어휘 집합을 **만들고(학습)**, `transformers`로 그 결과를 **사용(로드)**

### ❓ Q1-1. 패키지 역할 구분

**🔑 답:** (a) **tokenizers** (ByteLevelBPETokenizer) (b) **transformers** (GPT2Tokenizer)

### 📝 Q1-2. 빈칸 채우기 — 패키지 설치 (정답 포함)

In [ ]:
# 패키지 설치
!pip install -q korpora tokenizers transformers  # 🔑 정답: korpora, tokenizers, transformers

import tokenizers, transformers
print(f"tokenizers   버전: {tokenizers.__version__}")
print(f"transformers 버전: {transformers.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 2.7 MB/s eta 0:00:00
tokenizers   버전: 0.22.2
transformers 버전: 5.0.0


---

## 셀 2. 구글 드라이브 연동

토크나이저 학습 결과물(`vocab.json`, `merges.txt`)을 **구글 드라이브에 저장**합니다.
- Colab 런타임은 **임시 VM** — 세션 종료 시 모든 파일 삭제
- 구글 드라이브에 저장하면 런타임 종료 후에도 결과 유지
- 다음 실습(Lecture 04)에서 재사용

### ❓ Q2-1. 저장 경로 이해

**🔑 답:** (a) 이미 마운트되어 있어도 **강제 재연결**. 인증 오류 시 깔끔하게 재시도.
(b) 폴더가 **이미 존재할 때** `FileExistsError` 발생.

### 📝 Q2-2. 빈칸 채우기 — 드라이브 연동 (정답 포함)

In [ ]:
from google.colab import drive               # 🔑 정답: drive
drive.mount('/gdrive', force_remount=True)    # 🔑 정답: drive
print("✅ 구글 드라이브 연결 완료")

Mounted at /gdrive
✅ 구글 드라이브 연결 완료


In [ ]:
import os
BBPE_DIR = "/gdrive/My Drive/nlpbook/bbpe"
os.makedirs(BBPE_DIR, exist_ok=True)          # 🔑 정답: makedirs, exist_ok
print(f"✅ BBPE 저장 경로: {BBPE_DIR}")

✅ BBPE 저장 경로: /gdrive/My Drive/nlpbook/bbpe


### 🐛 Q2-3. 오류 수정 — 드라이브 경로

**🔑 답:**
1. `"/gdrive/nlpbook/bbpe"` → `"/gdrive/My Drive/nlpbook/bbpe"` — `My Drive` 누락 시 경로 없음
2. `os.makedirs(BBPE_DIR)` → `os.makedirs(BBPE_DIR, exist_ok=True)` — 이미 존재하면 에러

---
## 셀 3. 말뭉치 로딩 및 전처리

### 📚 NSMC(Naver Sentiment Movie Corpus)

| 항목 | 내용 |
|------|------|
| 총 데이터 수 | 약 200,000개 (train 150,000 / test 50,000) |
| 레이블 | 긍정(1) / 부정(0) |
| 특징 | 구어체, 비속어, 이모티콘, 오타 등 실제 인터넷 언어 포함 |

### 핵심: 토크나이저 학습은 **비지도(unsupervised)** 방식

텍스트의 통계적 패턴(어떤 글자쌍이 자주 함께 등장하는가)만 학습하므로 레이블이 **불필요**합니다.

### ❓ Q3-1. 말뭉치 이해

**🔑 답:** (a) `.text`는 리뷰 텍스트(문자열), `.label`은 감성 레이블(0 또는 1 정수)
(b) 토크나이저는 **텍스트의 바이그램 빈도**만 학습하는 비지도 방식이므로 정답 레이블 불필요.

### 📝 Q3-2. 빈칸 채우기 — 말뭉치 로딩 (정답 포함)

In [ ]:
from Korpora import Korpora                   # 🔑 정답: (a) Korpora (b) Korpora

print("NSMC 데이터를 다운로드합니다...")
nsmc = Korpora.load("nsmc", force_download=True)  # 🔑 정답: (c) Korpora (d) nsmc

print(f"\n✅ 학습 데이터: {len(nsmc.train):,}개")    # 🔑 정답: (e) train
print(f"✅ 테스트 데이터: {len(nsmc.test):,}개")

print("\n[데이터 예시 3개]")
for i in range(3):
    item = nsmc.train[i]
    label_str = "긍정" if item.label == 1 else "부정"
    print(f"  [{i}] ({label_str}) {item.text}")

NSMC 데이터를 다운로드합니다...

    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/



[nsmc] download ratings_train.txt: 14.6MB [00:00, 62.4MB/s]                            
[nsmc] download ratings_test.txt: 4.90MB [00:00, 37.4MB/s]                            



✅ 학습 데이터: 150,000개
✅ 테스트 데이터: 50,000개

[데이터 예시 3개]
  [0] (부정) 아 더빙.. 진짜 짜증나네요 목소리
  [1] (긍정) 흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나
  [2] (부정) 너무재밓었다그래서보는것을추천한다


---

### 💾 텍스트 파일로 저장하는 이유

`ByteLevelBPETokenizer.train()`은 **파일 경로 목록**(`files=[...]`)을 입력받습니다.
메모리의 리스트가 아닌 **디스크 파일**을 직접 읽어서 학습합니다. 저장 형식: 한 줄에 문장 하나.

### 📝 Q3-3. 빈칸 채우기 — 텍스트 저장 함수 (정답 포함)

In [ ]:
import os

def write_lines(path, lines):
    with open(path, 'w', encoding='utf-8') as f:          # 🔑 정답: (a) w (b) utf-8
        for line in lines:
            f.write(f'{line}\n')                           # 🔑 정답: (c) \n

TRAIN_PATH = "/root/train.txt"
TEST_PATH  = "/root/test.txt"

write_lines(TRAIN_PATH, nsmc.train.get_all_texts())       # 🔑 정답: (d) train (e) get_all_texts()
write_lines(TEST_PATH,  nsmc.test.get_all_texts())        # 🔑 정답: (f) test  (g) get_all_texts()

train_mb = os.path.getsize(TRAIN_PATH) / (1024 * 1024)
test_mb  = os.path.getsize(TEST_PATH)  / (1024 * 1024)
print(f"✅ train.txt 저장 완료 ({train_mb:.1f} MB)")
print(f"✅ test.txt  저장 완료 ({test_mb:.1f} MB)")

✅ train.txt 저장 완료 (12.5 MB)
✅ test.txt  저장 완료 (4.2 MB)


### 🐛 Q3-4. 오류 수정 — 데이터 저장

**🔑 답:**
1. `'r'`(읽기) → `'w'`(쓰기). 읽기 모드에서 write() 호출 시 에러.
2. `f'{line}'` → `f'{line}\n'`. 줄바꿈 없으면 모든 문장이 한 줄에 붙음.
3. `get_all_labels()` → `get_all_texts()`. 토크나이저 학습에는 **텍스트만** 필요.

### 🧮 Q3-5. 데이터 크기 계산

**🔑 답:** (a) **150,000개** (b) **200,000줄** (150,000 + 50,000)

---
## 셀 4. BBPE 어휘 집합 구축 (GPT 토크나이저)

### 🔬 Byte-Level BPE(BBPE)란?

| 특징 | 일반 BPE | BBPE |
|------|---------|------|
| 초기 어휘 크기 | 사용 문자 수 (한글 ~11,172) | **256개** (0x00~0xFF) |
| OOV 문제 | 있음 | **완전히 없음** |
| 사용 모델 | - | GPT-2, GPT-3, RoBERTa 등 |

### vocab.json vs merges.txt

| 파일 | 역할 |
|------|------|
| `vocab.json` | 완성된 어휘 집합. 토큰 → 인덱스 매핑 |
| `merges.txt` | 병합 규칙 목록. 토큰화 시 바이트 병합 순서 결정 |

> ⚠️ **두 파일 모두** 있어야 토크나이저 재로딩 가능

### ❓ Q4-1. BBPE 개념 이해

**🔑 답:** (a) **3바이트**
(b) 1바이트 = 0~255 → **256가지** → 초기 어휘 256개
(c) 모든 유니코드 문자는 1~4바이트 조합이며, 256개 바이트가 모두 초기 어휘에 포함되므로 **어떤 글자든 표현 가능**

### 📝 Q4-2. 빈칸 채우기 — BBPE 학습 코드 (정답 포함)

In [ ]:
from tokenizers import ByteLevelBPETokenizer  # 🔑 정답: (a) ByteLevelBPE

bbpe_tokenizer = ByteLevelBPETokenizer()      # 🔑 정답: (b) ByteLevelBPE

print("🔄 BBPE 토크나이저 학습 중... (약 1~2분)")
bbpe_tokenizer.train(                         # 🔑 정답: (c) train
    files=[TRAIN_PATH, TEST_PATH],            # 🔑 정답: (d) TEST_PATH
    vocab_size=10000,                          # 🔑 정답: (e) 10000
    min_frequency=2,                           # 🔑 정답: (f) 2
    special_tokens=["[PAD]"],                  # 🔑 정답: (g) [PAD]
)

bbpe_tokenizer.save_model(BBPE_DIR)           # 🔑 정답: (h) save_model

print(f"\n✅ 저장 완료: {BBPE_DIR}")
for fname in sorted(os.listdir(BBPE_DIR)):
    fpath = os.path.join(BBPE_DIR, fname)
    print(f"   📄 {fname}  ({os.path.getsize(fpath):,} bytes)")

🔄 BBPE 토크나이저 학습 중... (약 1~2분)

✅ 저장 완료: /gdrive/My Drive/nlpbook/bbpe
   📄 merges.txt  (153,014 bytes)
   📄 vocab.json  (212,853 bytes)
   📄 현재_나의조_최종  (4,096 bytes)


### 🐛 Q4-3. 오류 수정 — BBPE 학습

**🔑 답:**
1. `files="train.txt"` → `files=["train.txt"]` — **리스트**로 감싸야 함
2. `min_frequency=0` → `min_frequency=2` — 0이면 1회 등장 쌍도 병합 → 노이즈 증가
3. `special_tokens="[PAD]"` → `special_tokens=["[PAD]"]` — **리스트**로 감싸야 함
4. `bbpe.save(...)` → `bbpe.save_model(...)` — 메서드명이 `save_model`

### 🧮 Q4-4. 병합 횟수 계산

**🔑 답:** (a) 10000 - 256 - 1 = **9,743회**
(b) **0번** (special_tokens 리스트의 첫 번째 → 인덱스 0)

---

### 🔍 BBPE 토큰 표시 방식 이해하기 (매우 중요!)

`vocab.json`을 열어보면 한국어가 **깨진 문자처럼** 보입니다 — 이것은 오류가 아닙니다!

```
"안" → UTF-8: [0xEC, 0x95, 0x88]  (3바이트)
     → byte_encoder: 각 바이트를 유니코드 문자 1개로 매핑
     → BBPE 내부 표현: "ìĿ¨" (3개의 특수 문자)
```

`decode_bbpe_token()` 함수가 이 과정을 **역방향**으로 수행하여 한국어를 복원합니다.

| 함수 | 역할 |
|------|------|
| `bytes_to_unicode()` | 바이트(0~255) → 유니코드 문자 매핑 딕셔너리 생성 |
| `byte_encoder` | 위 함수의 결과. **바이트→유니코드** 방향 |
| `byte_decoder` | 역방향. **유니코드→바이트** 방향 |
| `decode_bbpe_token(token)` | BBPE 내부 표현 → 한국어 복원 |

### ❓ Q5-1. byte_encoder 이해

**🔑 답:** (a) **바이트→유니코드**
(b) **유니코드→바이트** (역방향)
(c) **3개** (바이트 1개 = 유니코드 문자 1개 매핑)

### 📝 Q5-2. 빈칸 채우기 — decode_bbpe_token 핵심 로직 (정답 포함)

In [ ]:
# 헬퍼 함수 정의
import json

def bytes_to_unicode():
    bs = (list(range(ord("!"), ord("~") + 1))
          + list(range(ord("¡"), ord("¬") + 1))
          + list(range(ord("®"), ord("ÿ") + 1)))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, [chr(c) for c in cs]))

byte_encoder = bytes_to_unicode()
byte_decoder = {v: k for k, v in byte_encoder.items()}

def decode_bbpe_token(token: str) -> str:
    try:
        byte_list = [byte_decoder[c] for c in token]      # 🔑 정답: (a) byte_decoder
    except KeyError:
        return token
    decoded = bytes(byte_list).decode("utf-8", errors="replace")  # 🔑 정답: (b) bytes
    if '\ufffd' in decoded:                                # 🔑 정답: (c) \ufffd
        return '[' + ' '.join(f'{b:02x}' for b in byte_list) + ']'
    return decoded.replace(' ', '▁')

def decode_merge_pair(part_a, part_b):
    return decode_bbpe_token(part_a + part_b)

# vocab.json 확인
with open(f"{BBPE_DIR}/vocab.json", 'r', encoding='utf-8') as f:
    vocab = json.load(f)
print(f"=== vocab.json: 총 {len(vocab):,}개 ===")
for i, (token, idx) in enumerate(vocab.items()):
    if i >= 8: break
    print(f"  {idx:5d}  {repr(token):20}  {decode_bbpe_token(token)}")

=== vocab.json: 총 10,000개 ===
      0  '[PAD]'               [PAD]
      1  '!'                   !
      2  '"'                   "
      3  '#'                   #
      4  '$'                   $
      5  '%'                   %
      6  '&'                   &
      7  "'"                   '


### 🐛 Q5-3. 오류 수정 — byte_encoder 관련

**🔑 답:**
1. `byte_decoder = byte_encoder` → `byte_decoder = {v: k for k, v in byte_encoder.items()}` — **역방향** 딕셔너리 생성 필요 (key-value 뒤집기)
2. `byte_encoder[c]` → `byte_decoder[c]` — 디코딩에는 **byte_decoder** 사용 (유니코드→바이트 방향)

### ❓ Q5-4. merges.txt 해석

**🔑 답:** (a) 초기 단계에서는 바이트가 합쳐져도 **완전한 UTF-8 문자가 되지 않음** (한글=3바이트 필요)
(b) 병합이 반복되면 **3바이트 이상이 모여** 완전한 한글 글자 완성

In [ ]:
# BBPE 동작 확인 — 바이트 인코딩 원리
sample = "안녕하세요 반갑습니다"

bytes_repr = ' '.join([f'{b:02x}' for b in sample.encode('utf-8')])
print(f"원문       : {sample}")
print(f"UTF-8 바이트: {bytes_repr}")
print(f"글자 수    : {len(sample)}자  →  바이트 수: {len(sample.encode('utf-8'))}바이트")

result = bbpe_tokenizer.encode(sample)
readable_tokens = [decode_bbpe_token(t) for t in result.tokens]

print(f"\n[ BBPE 토큰화 결과 ]")
for pos, (raw, readable) in enumerate(zip(result.tokens, readable_tokens), 1):
    print(f"  {pos:4d}   {raw:20}   {readable}")
print(f"\n  → 바이트 {len(sample.encode('utf-8'))}개가 {len(result.tokens)}개 토큰으로 병합됨")

원문       : 안녕하세요 반갑습니다
UTF-8 바이트: ec 95 88 eb 85 95 ed 95 98 ec 84 b8 ec 9a 94 20 eb b0 98 ea b0 91 ec 8a b5 eb 8b 88 eb 8b a4
글자 수    : 11자  →  바이트 수: 31바이트

[ BBPE 토큰화 결과 ]
     1   ìķĪë                   [ec 95 88 eb]
     2   ħķ                     [85 95]
     3   íķĺìĦ¸ìļĶ              하세요
     4   Ġë°ĺ                   ▁반
     5   ê°ĳ                    갑
     6   ìĬµëĭĪëĭ¤              습니다

  → 바이트 31개가 6개 토큰으로 병합됨


### 🧮 Q5-5. 바이트 수 계산

**🔑 답:** (a) 5 × 3 = **15바이트**
(b) 5 × 1 = **5바이트**
(c) 9(한글3×3) + 1(공백) + 4(영문4×1) = **14바이트**

---
## 셀 5. GPT 입력값 만들기

| 입력값 | 설명 |
|--------|------|
| `input_ids` | 토큰 인덱스 시퀀스. 각 토큰 → 어휘 집합의 인덱스(정수) |
| `attention_mask` | 실제 토큰=`1`, 패딩 토큰=`0`. 모델이 패딩을 무시하도록 알려줌 |

| 개념 | 동작 |
|------|------|
| 패딩(padding) | 짧은 문장 뒤에 `[PAD]` 추가하여 길이 맞춤 |
| 잘림(truncation) | `max_length` 초과 부분 잘라냄 |

### 📝 Q6-1. 빈칸 채우기 — GPT 토크나이저 로드 (정답 포함)

In [ ]:
from transformers import GPT2Tokenizer                 # 🔑 정답: (a) GPT2Tokenizer

tokenizer_gpt = GPT2Tokenizer.from_pretrained(BBPE_DIR)  # 🔑 정답: (b) GPT2Tokenizer
tokenizer_gpt.pad_token = "[PAD]"                      # 🔑 정답: (c) pad_token (d) [PAD]

print(f"✅ GPT 토크나이저 로드 완료")
print(f"   어휘 집합 크기 : {len(tokenizer_gpt):,}개")
print(f"   패딩 토큰      : '{tokenizer_gpt.pad_token}' (id={tokenizer_gpt.pad_token_id})")

✅ GPT 토크나이저 로드 완료
   어휘 집합 크기 : 10,001개
   패딩 토큰      : '[PAD]' (id=0)


### 🐛 Q6-2. 오류 수정 — 토크나이저 로드

**🔑 답:**
1. `BertTokenizer` → `GPT2Tokenizer` — BBPE는 GPT2Tokenizer로 로드
2. `BertTokenizer` → `GPT2Tokenizer` (연동)
3. `tokenizer.pad_token = "[PAD]"` 누락 — 없으면 패딩 시 `ValueError` 발생

### ❓ Q6-3. GPT2Tokenizer 이해

**🔑 답:** (a) `vocab.json`과 `merges.txt`
(b) GPT는 **자기회귀(autoregressive)** 학습으로 고정 길이 사용 → 패딩 불필요. 파인튜닝/배치 처리 시에만 필요.

In [ ]:
# 예시 문장 토큰화
sentences = [
    "아 더빙.. 진짜 짜증나네요 목소리",
    "흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나",
    "별루 였다..",
]

print("=== GPT 토크나이저 토큰화 결과 ===")
for i, sent in enumerate(sentences, 1):
    tokens  = tokenizer_gpt.tokenize(sent)
    readable = [decode_bbpe_token(t) for t in tokens]
    print(f"\n[문장{i}] {sent}")
    print(f"  토큰 수: {len(tokens)}")
    print(f"  한국어 : {readable}")

=== GPT 토크나이저 토큰화 결과 ===

[문장1] 아 더빙.. 진짜 짜증나네요 목소리
  토큰 수: 7
  한국어 : ['아', '▁더빙', '..', '▁진짜', '▁짜증나', '네요', '▁목소리']

[문장2] 흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나
  토큰 수: 15
  한국어 : ['흠', '...', '포스터', '보고', '▁초딩', '영화', '줄', '....', '오버', '연기', '조차', '▁가볍', '지', '▁않', '구나']

[문장3] 별루 였다..
  토큰 수: 4
  한국어 : ['별루', '[20 ec 98]', '[80 eb 8b a4]', '..']


### 🧮 Q6-4. 토큰화 결과 해석

**🔑 답:** (실행 결과에 따라 다름) (a)(b) 출력 확인
(c) 토큰 수가 12 초과인 문장 — 문장2가 가장 길어 잘림 가능성 높음

### 📝 Q6-5. 빈칸 채우기 — 배치 인코딩 (정답 포함)

In [ ]:
batch_gpt = tokenizer_gpt(
    sentences,                 # 🔑 정답: (a) sentences
    padding="max_length",      # 🔑 정답: (b) max_length
    max_length=12,             # 🔑 정답: (c) 12
    truncation=True,           # 🔑 정답: (d) True
)

### 🐛 Q6-6. 오류 수정 — 배치 인코딩

**🔑 답:**
1. `padding="longest"` → `padding="max_length"` — `"longest"`는 배치 내 최장 문장 기준으로 shape이 매번 달라짐
2. `truncation=False` → `truncation=True` — False이면 max_length 초과 시 잘리지 않아 에러 가능

In [ ]:
# GPT 모델 입력 생성 (정상 코드)
batch_gpt = tokenizer_gpt(
    sentences,
    padding="max_length",
    max_length=12,
    truncation=True,
)

header = "구분    " + "  ".join([f"토큰{i:02d}" for i in range(1, 13)])
sep    = "─" * 82

print("=== GPT input_ids ===")
print("  0 = [PAD] 토큰")
print(); print(header); print(sep)
for i, ids in enumerate(batch_gpt['input_ids'], 1):
    print(f"문장{i}  " + "  ".join([f"{x:5d}" for x in ids]))

print()
print("=== GPT attention_mask ===")
print("  1 = 실제 토큰,  0 = 패딩")
print(); print(header); print(sep)
for i, mask in enumerate(batch_gpt['attention_mask'], 1):
    print(f"문장{i}  " + "  ".join([f"{x:5d}" for x in mask]))

=== GPT input_ids ===
  0 = [PAD] 토큰

구분    토큰01  토큰02  토큰03  토큰04  토큰05  토큰06  토큰07  토큰08  토큰09  토큰10  토큰11  토큰12
──────────────────────────────────────────────────────────────────────────────────
문장1    334   2338    263    581   4055    464   3808      0      0      0      0      0
문장2   3693    336   2876    758   2883    356    806    422   9875    875   2960   7292
문장3   4957    451   3653    263      0      0      0      0      0      0      0      0

=== GPT attention_mask ===
  1 = 실제 토큰,  0 = 패딩

구분    토큰01  토큰02  토큰03  토큰04  토큰05  토큰06  토큰07  토큰08  토큰09  토큰10  토큰11  토큰12
──────────────────────────────────────────────────────────────────────────────────
문장1      1      1      1      1      1      1      1      0      0      0      0      0
문장2      1      1      1      1      1      1      1      1      1      1      1      1
문장3      1      1      1      1      0      0      0      0      0      0      0      0


### 🧮 Q6-7. input_ids와 attention_mask 해석

**🔑 답:** (a) 문장이 **max_length(12) 이상**이어서 truncation 발생 → 패딩 없이 모두 실제 토큰
(b) 12 - 4 = **8개**

### 📝 Q6-8. 빈칸 채우기 — attention_mask 직접 작성 (정답 포함)

### 📝 Q6-8. 빈칸 채우기 — attention_mask 직접 작성

**🔑 답:**
- input_ids 빈칸: **0, 0, 0** ([PAD] 토큰의 인덱스)
- attention_mask: `[1, 1, 1, 1, 1, 0, 0, 0]` (실제 5개=1, 패딩 3개=0)

### 🧮 Q6-9. Shape 계산

**🔑 답:** (a) **(3 × 12)**
(b) **(5 × 20)**

---
## 📝 강의 요약

| 항목 | 내용 |
|------|------|
| **서브워드 토큰화** | 어휘 집합 크기를 적절히 유지하면서 OOV 문제 방지 |
| **BPE** | 가장 자주 등장하는 바이그램 쌍을 **빈도** 기준으로 반복 병합 |
| **BBPE** | UTF-8 **바이트** 단위에서 시작. OOV 완전 제거. GPT 계열 사용 |
| **vocab.json** | 완성된 어휘 집합. 토큰 → 인덱스 |
| **merges.txt** | 병합 규칙. 토큰화 시 병합 순서 결정 |
| **byte_encoder** | 바이트(0~255) → 유니코드 문자 매핑 |
| **GPT 입력** | `input_ids` + `attention_mask` |

---
## 📋 종합 퀴즈 — 교수용 답안

### Quiz 1. **🔑 답:** 단어: 크다/있다, 문자: 작다/없다, 서브워드: 중간/거의 없다

### Quiz 2. **🔑 답:**
(a) (a,b)=8, (b,c)=7, (b,d)=3, (c,d)=2
(b) (a,b)=8회 → 새 어휘: **{a, b, c, d, ab}**

### Quiz 3. **🔑 답:** 10000 - 256 - 1 = **9,743회**

### Quiz 4. **🔑 답:** (a) **9** (b) **4** (c) **14** (9+1+4)

### Quiz 5. **🔑 답:**
(a) ByteLevelBPETokenizer (b) GPT2Tokenizer (c) ByteLevelBPETokenizer
(d) train (e) save_model (f) GPT2Tokenizer (g) pad_token (h) truncation

### Quiz 6. **🔑 답:**
① BertTokenizer → GPT2Tokenizer
② files="data.txt" → files=["data.txt"]
③ special_tokens="[PAD]" → special_tokens=["[PAD]"]
④ save() → save_model()
⑤ BertTokenizer → GPT2Tokenizer
⑥ truncation=True 누락 + tok.pad_token="[PAD]" 누락

### Quiz 7. **🔑 답:** `[1,1,1,1,1,1,0,0,0,0]`

### Quiz 8. **🔑 답:** (a) 토큰화 불가 (병합 순서 소실) (b) **자주**

### Quiz 9. **🔑 답:** (a) **256** (b) **Ġ** (U+0120)

### Quiz 10. **🔑 답:** ③→②→④→⑤→⑦→⑥→⑧→①

### Quiz 11. **🔑 답:** (a) **5** (b) **0** (마지막은 패딩)

### Quiz 12. **🔑 답:** (a) **빈도** (b) **우도(likelihood)** (c) GPT→**BPE(BBPE)**, BERT→**WordPiece**